# Gemma 4 31B on vXPU, from a notebook with no GPU

This notebook runs the examples from the Hugging Face
[Gemma 4 model docs](https://huggingface.co/docs/transformers/en/model_doc/gemma4)
against `google/gemma-4-31B-it` **without a local GPU, CUDA, or any
downloaded weights**. The machine running this notebook is a thin client:

1. It instantiates the model on torch's `meta` device and exports a
   weightless artifact (manifest + two `torch.export` graphs, ~75 MB).
2. It ships the artifact to the **vXPU router** in a Kubernetes cluster.
3. The router decides *where* the model runs. Gemma 4 31B is 62.6 GB in
   bf16, more than a 24 GB NVIDIA L4 can hold, so on an L4 cluster the
   router sizes a CPU executor (~75 GiB RAM) instead; a model that fits
   (e.g. Gemma 4 E4B, 16 GB) goes to the GPU. The executor rehydrates
   the weights straight from the Hub with HTTP range reads.
4. The notebook chats with the model over gRPC. The HF `generate()` calls
   below become vXPU session calls.

Where the HF examples need `model.generate(...)` on a local accelerator,
the vXPU version is `session.chat(...)` / `session.generate(...)`.

## Prerequisites

- `kubectl` pointing at a GKE cluster (any node pool works; a node with
  enough RAM for the model is needed when it does not fit a GPU).
- The vXPU images and CLI:

```sh
cd vxpu
gcloud builds submit --config cloudbuild.yaml \
    --substitutions _IMAGE=gcr.io/$PROJECT/vxpu-executor:v8 .
gcloud builds submit --config cloudbuild-router.yaml \
    --substitutions _IMAGE=gcr.io/$PROJECT/vxpu-router:v2 .
export VXPU_EXECUTOR_IMAGE=gcr.io/$PROJECT/vxpu-executor:v8
export VXPU_ROUTER_IMAGE=gcr.io/$PROJECT/vxpu-router:v2
go build -o bin/vxpu ./cmd/vxpu
```

- This notebook's Python environment (no CUDA needed):

```sh
uv venv -p 3.13 .venv && uv pip install -e python/ jupyter
```

In [1]:
import json, os, shutil, subprocess, time

import torch
from vxpu.client import Client, PortForward
from vxpu.export import export_artifact

MODEL_ID = "google/gemma-4-31B-it"
ARTIFACT_DIR = os.environ.get("VXPU_ARTIFACT_DIR", "artifacts/gemma-4-31b-it")
VXPU_BIN = os.environ.get("VXPU_BIN", shutil.which("vxpu") or "../../bin/vxpu")
MAX_CACHE_LEN = 2048  # tokens per session (prompt + reply), fixed at export

print("CUDA available on this machine:", torch.cuda.is_available())

CUDA available on this machine: False


## 1. Export the artifact (thin client, no weights)

`export_artifact` builds a content-addressed manifest from Hub metadata
(per-file sha256 and range-fetched safetensors headers), instantiates the
model on the `meta` device, and captures `prefill`/`decode` graphs with
`torch.export`. The 31B model exports on a laptop in a couple of minutes
because nothing is materialized.

In [2]:
if not os.path.exists(os.path.join(ARTIFACT_DIR, "decode.pt2")):
    started = time.time()
    manifest, binding = export_artifact(MODEL_ID, ARTIFACT_DIR, max_cache_len=MAX_CACHE_LEN)
    print(f"exported in {time.time() - started:.0f}s")
else:
    manifest = json.load(open(os.path.join(ARTIFACT_DIR, "manifest.json")))
    binding = json.load(open(os.path.join(ARTIFACT_DIR, "binding.json")))

weights_gb = sum(f["size"] for f in manifest["files"].values()) / 1e9
artifact_mb = sum(os.path.getsize(os.path.join(ARTIFACT_DIR, f)) for f in os.listdir(ARTIFACT_DIR)) / 1e6
print(f"{MODEL_ID}: {len(manifest['tensors'])} tensors, {weights_gb:.1f} GB of weights referenced")
print(f"artifact on disk: {artifact_mb:.0f} MB "
      f"({len(binding['bound'])} bound, {len(binding['derived'])} derived, {len(binding['state'])} state tensors)")

google/gemma-4-31B-it: 1188 tensors, 62.5 GB of weights referenced
artifact on disk: 75 MB (1189 bound, 3 derived, 180 state tensors)


## 2. Bring up the router and connect

`vxpu up` creates the router pod (plus its ServiceAccount/Role and a
Service) if it is not already running. From outside the cluster we reach
it through `kubectl port-forward`; a notebook running *inside* the
cluster would connect to `vxpu-router:50051` directly.

In [3]:
print(subprocess.run([VXPU_BIN, "up"], check=True, capture_output=True, text=True).stdout)

forward = PortForward("pod/vxpu-router")
address = forward.start()
client = Client(address, timeout=3600)
print("connected to vxpu-router via", address)

pod/vxpu-router condition met
router vxpu-router is ready
  from outside the cluster: kubectl port-forward pod/vxpu-router 50051:50051
  from inside the cluster:  vxpu-router:50051



connected to vxpu-router via 127.0.0.1:58512


## 3. Load the model where it fits

`load_artifact` ships the ~75 MB artifact. The router caches the weight
files, picks a placement, creates the executor pod, and returns; the
executor then rehydrates the weights asynchronously while we poll for
readiness. A cold load of 62.6 GB takes on the order of 10–20 minutes
(download bound); a warm executor answers immediately.

In [4]:
session = client.load_artifact(ARTIFACT_DIR)

shipping artifact /tmp/vxpu-artifacts/gemma-4-31b-it (74 MB of graphs; weights stay remote)


executor placed and artifact delivered (554s); rehydrating weights...
  loading... (554s)


  loading... (569s)


  loading... (585s)


  loading... (600s)


...

  loading... (690s)


  loading... (705s)


  loading... (720s)


  loading... (735s)


model ready in 750s


In [5]:
# Where did the router put it?
print(subprocess.run(
    ["kubectl", "get", "pods", "-l", "app=vxpu-executor", "-o",
     "custom-columns=POD:.metadata.name,NODE:.spec.nodeName,CPU:.spec.containers[0].resources.requests.cpu,"
     "MEMORY:.spec.containers[0].resources.requests.memory,GPU:.spec.containers[0].resources.limits.nvidia\\.com/gpu"],
    capture_output=True, text=True).stdout)

POD                                              NODE                                                  CPU   MEMORY        GPU
vxpu-executor-092821c865aeaaa1481ead907c24da18   gke-ai-us-central1-b-c3dstandard60lss-cad71bbc-wv98   30    79256664634   <none>



## 4. The Hugging Face examples, through vXPU

### Causal language modeling

HF docs:

```python
from transformers import AutoTokenizer, Gemma4ForCausalLM

model = Gemma4ForCausalLM.from_pretrained("google/gemma-4-E2B-it")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-E2B-it")

prompt = "What is your favorite condiment?"
inputs = tokenizer(prompt, return_tensors="pt")
generate_ids = model.generate(inputs.input_ids, max_length=30)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
```

With vXPU there is no `from_pretrained`: the model is already resident on
the executor. `session.chat` applies the chat template on the executor
and returns the reply (greedy decoding; the executor stops on any of the
model's end-of-turn ids).

In [6]:
prompt = "What is your favorite condiment?"
reply = session.chat(prompt, max_new_tokens=96)
reply

As an AI, I don’t have taste buds, so I can’t enjoy the flavor of condiments. However, based on the data I process, **Sriracha** and **Mayonnaise** seem to be some of the most debated and beloved options!

If I had to choose one based on "personality," I’d go with **Hot Sauce**—because it adds a bit of spark to everything! 

How about you? What's your

*(96 tokens, 1369 ms/token on the executor, prefill 1532 ms, 115 tokens in session)*

### Function calling

HF docs render the chat template with `tools=[...]` and call
`model.generate` on the result. vXPU's `raw_prompt` mode does the same:
the client renders the prompt and the executor tokenizes it verbatim.
The docs use `AutoProcessor`; its Gemma 4 processor also wraps the image
and audio front-ends (pulling in `pillow`/`torchvision`), so this text-only
notebook renders the identical template with `AutoTokenizer`, which
downloads only the tokenizer files, no weights.

In [7]:
from transformers import AutoTokenizer

WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_n_day_weather_forecast",
        "description": "Get an N-day weather forecast",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "format": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The temperature unit to use",
                },
                "num_days": {
                    "type": "integer",
                    "description": "The number of days to forecast",
                },
            },
            "required": ["location", "format", "num_days"],
        },
    },
}

messages = [
    {
        "role": "user",
        "content": "What's the weather like the next 3 days in San Francisco, CA (using F)?",
    },
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")

text = tokenizer.apply_chat_template(
    messages,
    tools=[WEATHER_TOOL],
    tokenize=False,
    add_generation_prompt=True,
)

tool_session = client.new_session()
tool_call = tool_session.generate(text, max_new_tokens=200)
print(tool_call)

<|tool_call>call:get_n_day_weather_forecast{format:<|"|>fahrenheit<|"|>,location:<|"|>San Francisco, CA<|"|>,num_days:3}<tool_call|>


The model answers with a tool call in Gemma 4's native format,
`<|tool_call>call:name{arg:<|"|>value<|"|>,...}<tool_call|>`, then stops:
the next token would be `<|tool_response>`, one of the model's end ids,
which the executor reads from `generation_config.json`. Raw-mode replies
keep the control tokens so the caller can parse them. Completing
the loop is the same as with HF: append the call as a structured
`tool_calls` message plus a `tool` response, re-render the template, and
generate again. The executor keeps the session's KV cache and prefills
only from the first token that differs, so the second turn does not pay
for the tool declarations again (compare `new_prompt_tokens` with
`session_tokens`).

In [8]:
import re

def parse_gemma_tool_call(text):
    # Parse the first <|tool_call>call:name{k:<|"|>v<|"|>,k2:3}<tool_call|> block.
    m = re.search(r"<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>", text, re.S)
    if not m:
        return None
    name, body = m.group(1), m.group(2)
    args = {}
    for key, quoted, bare in re.findall(r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,]+))', body, re.S):
        if quoted:
            args[key] = quoted
        elif re.fullmatch(r"-?\d+(\.\d+)?|true|false|null", bare.strip()):
            args[key] = json.loads(bare.strip())
        else:
            args[key] = bare.strip()
    return {"name": name, "arguments": args}

call = parse_gemma_tool_call(tool_call) or {
    "name": "get_n_day_weather_forecast",
    "arguments": {"location": "San Francisco, CA", "format": "fahrenheit", "num_days": 3},
}
print("parsed:", call)

messages.append({"role": "assistant", "tool_calls": [{"id": "call_1", "type": "function", "function": call}]})
messages.append({
    "role": "tool",
    "tool_call_id": "call_1",
    "name": call["name"],
    "content": json.dumps({
        "location": "San Francisco, CA",
        "format": "fahrenheit",
        "forecast": [
            {"day": 1, "high": 68, "low": 54, "conditions": "sunny"},
            {"day": 2, "high": 65, "low": 53, "conditions": "fog in the morning"},
            {"day": 3, "high": 62, "low": 52, "conditions": "light rain"},
        ],
    }),
})

text = tokenizer.apply_chat_template(
    messages,
    tools=[WEATHER_TOOL],
    tokenize=False,
    add_generation_prompt=True,
)
final = tool_session.chat(text, max_new_tokens=160, raw_prompt=True)
final

parsed: {'name': 'get_n_day_weather_forecast', 'arguments': {'format': 'fahrenheit', 'location': 'San Francisco, CA', 'num_days': 3}}


<|channel>thought
<channel|>The weather forecast for San Francisco, CA over the next 3 days is as follows:

*   **Day 1:** High of 68°F, low of 54°F, and sunny.
*   **Day 2:** High of 65°F, low of 53°F, with fog in the morning.
*   **Day 3:** High of 62°F, low of 52°F, with light rain.

*(107 tokens, 1190 ms/token on the executor, prefill 4523 ms, 427 tokens in session)*

### Multi-turn chat

Not in the HF docs, but it is what a KV-cached session is for: a
follow-up turn re-uses the cache and prefills only from the first token
that differs from what is cached. (With Gemma 4's template that is a
little more than the new message: the generation prompt carries an
empty `thought` channel that the re-rendered transcript omits, so the
previous reply is re-prefilled too; the shared prefix is reused.)

In [9]:
turn1 = session.chat("Give me three facts about Kubernetes, one sentence each.", max_new_tokens=120)
print(turn1.text)
print(f"\n[{turn1.new_prompt_tokens} new prompt tokens prefilled, session now {turn1.session_tokens} tokens]")

Kubernetes is an open-source orchestration system designed to automate the deployment, scaling, and management of containerized applications.

It was originally developed by Google and is now maintained by the Cloud Native Computing Foundation (CNCF).

The system uses a declarative approach, allowing users to define a desired state that Kubernetes then works to maintain automatically.

[121 new prompt tokens prefilled, session now 204 tokens]


In [10]:
turn2 = session.chat("Which of those three matters most for running AI workloads? Answer briefly.", max_new_tokens=96)
print(turn2.text)
print(f"\n[{turn2.new_prompt_tokens} new prompt tokens prefilled, session now {turn2.session_tokens} tokens]")

The first fact—**automating deployment, scaling, and management**—matters most, as AI workloads require the ability to dynamically scale massive compute resources (like GPUs) to handle heavy processing demands.

[96 new prompt tokens prefilled, session now 269 tokens]


### Configuration

The HF docs' configuration example builds a `Gemma4ForConditionalGeneration`
from configs. It is pure architecture code and needs no GPU; we run it on
the `meta` device so it also needs no RAM (the default text config is
sized like a real model), which is exactly how the export above works.

In [11]:
from transformers import (
    Gemma4AudioConfig,
    Gemma4Config,
    Gemma4ForConditionalGeneration,
    Gemma4TextConfig,
    Gemma4VisionConfig,
)

# Initializing a Gemma 4 Audio config.
audio_config = Gemma4AudioConfig()

# Initializing a Gemma 4 Text config.
text_config = Gemma4TextConfig()

# Initializing a Gemma 4 vision config.
vision_config = Gemma4VisionConfig()

# Initializing a Gemma 4 config similar to google/gemma-4-e2b-it
# (keyword arguments: transformers 5.x no longer accepts them positionally)
configuration = Gemma4Config(text_config=text_config, vision_config=vision_config, audio_config=audio_config)

# Initializing a model from the google/gemma-4-e2b-it configuration (weightless: meta device)
with torch.device("meta"):
    model = Gemma4ForConditionalGeneration(configuration)

# Accessing the model configuration
configuration = model.config
print(type(model).__name__, "with", sum(p.numel() for p in model.parameters()) / 1e9, "B parameters (on", next(model.parameters()).device, ")")

Gemma4ForConditionalGeneration with 5.554675488 B parameters (on meta )


### Image and audio examples

The HF page also shows image-text-to-text (pipeline and `AutoModel`)
and audio transcription. They are not run here:

- The exported artifact captures the **text** decoder path
  (`input_ids` → logits), so the vision tower is not part of the graphs
  the executor runs. Multimodal export is a vXPU roadmap item.
- Audio is native only on the E2B/E4B sizes; the 31B has no audio
  backbone (`audio_config: null`).

## 5. Clean up

The executor evicts the model after it has been idle (default 300 s) but
keeps the pod; delete the router and executor when you are done:

```sh
vxpu down
kubectl delete pods -l app=vxpu-executor
```

In [12]:
forward.stop()
client.close()